# 03 Búsqueda vectorial con TF-IDF

## Objetivo

- Calcular e interpretar TF, DF, IDF y TF-IDF.
- Implementar TF-IDF mediante operaciones explícitas y verificarlo con `TfidfVectorizer`.
- Calcular similitud coseno mediante el producto punto.
- Implementar BM25 y comprender sus componentes.
- Construir y comparar rankings para una misma consulta.

### 1. Recuperar y revisar el corpus

Utilice el corpus de Project Gutenberg y el preprocesamiento de la práctica anterior.

Construya una tabla con:

**Identificador de documento | Título | Número de términos únicos**

Presente:

- Número de documentos.
- Tamaño del vocabulario.
- Longitud mínima, media y máxima de los documentos.

Conserve el texto original para mostrar fragmentos de los resultados. Todas las técnicas deben utilizar **los mismos documentos y términos procesados**.

In [2]:
import re
import tarfile
import zipfile
import numpy as np
import pandas as pd

from pathlib import Path


In [3]:
# Ubicación del corpus utilizado en la práctica anterior

CARPETA = Path.cwd()
RUTA_ZIP = CARPETA.parent / "02" / "txt-files.tar.zip"

# Mismo preprocesamiento del notebook 02
def tokenizar(texto):
    texto = texto.lower()
    return re.findall(r'\b[a-z0-9]+\b', texto)

# Seleccionar un subconjunto del corpus
TAMANO_CORPUS = 100

documentos_originales = {}

with zipfile.ZipFile(RUTA_ZIP, "r") as zf:
    nombres_zip = zf.namelist()

    tar_interno = next(
        (n for n in nombres_zip if n.lower().endswith(".tar")),
        None
    )

    if tar_interno:
        with zf.open(tar_interno) as archivo_tar:
            with tarfile.open(fileobj=archivo_tar, mode="r|") as tf:
                for miembro in tf:
                    if miembro.isfile() and miembro.name.lower().endswith(".txt"):
                        nombre = miembro.name

                        if nombre in documentos_originales:
                            continue

                        archivo = tf.extractfile(miembro)
                        if archivo is None:
                            continue

                        texto = archivo.read().decode("utf-8", errors="ignore")
                        documentos_originales[nombre] = texto

                        if len(documentos_originales) >= TAMANO_CORPUS:
                            break
    else:
        nombres_txt = sorted(
            n for n in nombres_zip if n.lower().endswith(".txt")
        )

        for nombre in nombres_txt[:TAMANO_CORPUS]:
            with zf.open(nombre) as archivo:
                documentos_originales[nombre] = archivo.read().decode(
                    "utf-8", errors="ignore"
                )

# Tokenizar los documentos
documentos = {
    nombre: tokenizar(texto)
    for nombre, texto in documentos_originales.items()
}

# Construir tabla de documentos
tabla_documentos = pd.DataFrame([
    {
        "Identificador": nombre,
        "Título": Path(nombre).stem,
        "Términos únicos": len(set(tokens))
    }
    for nombre, tokens in documentos.items()
])

display(tabla_documentos)

# Estadísticas generales
vocabulario = set()
longitudes = []

for tokens in documentos.values():
    vocabulario.update(tokens)
    longitudes.append(len(tokens))

print("Número de documentos:", len(documentos))
print("Tamaño del vocabulario:", len(vocabulario))
print("Longitud mínima:", min(longitudes))
print("Longitud media:", np.mean(longitudes))
print("Longitud máxima:", max(longitudes))

,Identificador,Título,Términos únicos
0,cache/epub/10000/pg10000.txt,pg10000,1870
1,cache/epub/10001/pg10001.txt,pg10001,2108
2,cache/epub/10002/pg10002.txt,pg10002,5306
3,cache/epub/10003/pg10003.txt,pg10003,6121
4,cache/epub/10004/pg10004.txt,pg10004,6817
...,...,...,...
95,cache/epub/10087/pg10087.txt,pg10087,5304
96,cache/epub/10088/pg10088.txt,pg10088,11100
97,cache/epub/10089/pg10089.txt,pg10089,3610
98,cache/epub/1008/pg1008.txt,pg1008,15200


Número de documentos: 100
Tamaño del vocabulario: 98167
Longitud mínima: 5400
Longitud media: 57156.29
Longitud máxima: 182221


### 2. Implementación de Funciones

Programe funciones para obtener:

1. TF de cada término en cada documento.
2. DF de cada término en **todo el corpus**.
3. IDF de cada término.
4. Los seis pesos TF-IDF correspondientes.

Para facilitar la comparación posterior con scikit-learn, utilizaremos:

$TF(t,d)=f(t,d)$

$
IDF(t)=\ln\left(\frac{N}{DF(t)}\right)+1
$

$
w_{t,d}=TF(t,d)\times IDF(t)
$

Aquí, $f(t,d)$ es el número de apariciones y $N$ el número de documentos.

**Nota:** Ahora utilizamos logaritmo natural y sumamos uno al IDF. Esta es la fórmula de scikit-learn cuando `smooth_idf=False`. [sklearn](https://sklearn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfTransformer.html?utm_source=chatgpt.com)

In [4]:
from collections import Counter
import math

In [5]:
# Número total de documentos
N = len(documentos)

# Calcular TF de cada documento
tf_documentos = {}

for nombre, tokens in documentos.items():
    tf_documentos[nombre] = Counter(tokens)

# Calcular DF de cada término en todo el corpus
df_terminos = Counter()

for nombre, tokens in documentos.items():
    df_terminos.update(set(tokens))

# Función para calcular TF
def calcular_tf(termino, documento):
    return tf_documentos[documento].get(termino, 0)

# Función para calcular DF
def calcular_df(termino):
    return df_terminos.get(termino, 0)

# Función para calcular IDF
def calcular_idf(termino):
    df = calcular_df(termino)

    if df == 0:
        return 0.0

    return math.log(N / df) + 1

# Función para calcular TF-IDF
def calcular_tfidf(termino, documento):
    tf = calcular_tf(termino, documento)
    idf = calcular_idf(termino)

    return tf * idf

# Seleccionar un documento y seis términos para comprobar
documento_ejemplo = list(documentos.keys())[0]
terminos_ejemplo = list(tf_documentos[documento_ejemplo].keys())[:6]

resultados = []

for termino in terminos_ejemplo:
    resultados.append({
        "Término": termino,
        "TF": calcular_tf(termino, documento_ejemplo),
        "DF": calcular_df(termino),
        "IDF": calcular_idf(termino),
        "TF-IDF": calcular_tfidf(termino, documento_ejemplo)
    })

tabla_tfidf = pd.DataFrame(resultados)

print("Documento:", documento_ejemplo)
display(tabla_tfidf)


Documento: cache/epub/10000/pg10000.txt


,Término,TF,DF,IDF,TF-IDF
0,the,1144,100,1.000000,1144.000000
1,project,88,100,1.000000,88.000000
2,gutenberg,98,100,1.000000,98.000000
3,ebook,13,100,1.000000,13.000000
4,of,902,100,1.000000,902.000000
5,magna,13,9,3.407946,44.303293


### 3. Implementar TF-IDF para el corpus

1. Construir un vocabulario con un orden definido.
2. Contar las frecuencias TF.
3. Calcular DF contando cada documento una sola vez por término.
4. Calcular IDF con la función de la sección anterior.
5. Obtener la matriz documento-término TF-IDF.

In [6]:
from scipy.sparse import csr_matrix

In [7]:
# Construir el vocabulario ordenado
vocabulario_ordenado = sorted(df_terminos.keys())

# Crear un índice para cada término
indice_terminos = {
    termino: i
    for i, termino in enumerate(vocabulario_ordenado)
}

# Mantener el orden de los documentos
nombres_documentos = list(documentos.keys())

# Calcular IDF para todos los términos
idf_terminos = {
    termino: calcular_idf(termino)
    for termino in vocabulario_ordenado
}

# Construir la matriz TF-IDF
filas = []
columnas = []
valores = []

for i, nombre in enumerate(nombres_documentos):
    for termino, frecuencia in tf_documentos[nombre].items():

        j = indice_terminos[termino]
        tfidf = frecuencia * idf_terminos[termino]

        filas.append(i)
        columnas.append(j)
        valores.append(tfidf)

# Crear matriz dispersa
matriz_tfidf = csr_matrix(
    (valores, (filas, columnas)),
    shape=(len(nombres_documentos), len(vocabulario_ordenado))
)

# Mostrar información de la matriz
print("Dimensiones de la matriz:", matriz_tfidf.shape)
print("Número de valores distintos de cero:", matriz_tfidf.nnz)
print("Tipo de matriz:", type(matriz_tfidf))

# Mostrar una pequeña parte de la matriz
tabla_matriz = pd.DataFrame(
    matriz_tfidf[:5, :10].toarray(),
    index=nombres_documentos[:5],
    columns=vocabulario_ordenado[:10]
)

display(tabla_matriz)


Dimensiones de la matriz: (100, 98167)
Número de valores distintos de cero: 603074
Tipo de matriz: <class 'scipy.sparse._csr.csr_matrix'>


,0,00,000,001,0014,0023,003,0035,004,0042
cache/epub/10000/pg10000.txt,12.160883,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
cache/epub/10001/pg10001.txt,0.000000,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
cache/epub/10002/pg10002.txt,0.000000,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
cache/epub/10003/pg10003.txt,0.000000,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
cache/epub/10004/pg10004.txt,0.000000,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### 4. Verificar con `TfidfVectorizer`

Construya una segunda representación utilizando la librería `TfidfVectorizer`, con esta configuración:

| Parámetro | Valor |
|---|---|
| `use_idf` | `True` |
| `smooth_idf` | `False` |
| `sublinear_tf` | `False` |
| `norm` | `None` |

Esta configuración permite comparar pesos sin normalización y con TF como conteo directo. [scikit-learn 1.9.1 documentation](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html?highlight=vectorizer\&utm_source=chatgpt.com)

Aseguren que el vectorizador recibe exactamente los mismos tokens que su implementación.

Calcule la **máxima diferencia absoluta** entre ambas matrices.

Si no coinciden, investiguen el preprocesamiento, el orden de columnas, la fórmula de IDF y la normalización.

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [9]:
# Crear el vectorizador con la configuración solicitada
vectorizador = TfidfVectorizer(
    use_idf=True,
    smooth_idf=False,
    sublinear_tf=False,
    norm=None,
    tokenizer=lambda texto: texto,
    preprocessor=lambda texto: texto,
    token_pattern=None,
    lowercase=False,
    vocabulary=indice_terminos
)

# Recuperar los mismos tokens y el mismo orden de documentos
corpus_tokens = [
    documentos[nombre]
    for nombre in nombres_documentos
]

# Construir la matriz TF-IDF con scikit-learn
matriz_sklearn = vectorizador.fit_transform(corpus_tokens)

# Comparar las dimensiones
print("Matriz manual:", matriz_tfidf.shape)
print("Matriz scikit-learn:", matriz_sklearn.shape)

# Calcular la diferencia entre ambas matrices
diferencia = matriz_tfidf - matriz_sklearn

if diferencia.nnz == 0:
    diferencia_maxima = 0.0
else:
    diferencia_maxima = np.max(np.abs(diferencia.data))

print("Máxima diferencia absoluta:", diferencia_maxima)

# Comprobar si las matrices coinciden
tolerancia = 1e-10

if diferencia_maxima < tolerancia:
    print("Las matrices TF-IDF coinciden correctamente.")
else:
    print("Las matrices presentan diferencias.")

# Comparar algunos valores
comparacion = pd.DataFrame({
    "Término": vocabulario_ordenado[:10],
    "TF-IDF manual": matriz_tfidf[0, :10].toarray().flatten(),
    "TF-IDF scikit-learn": matriz_sklearn[0, :10].toarray().flatten()
})

display(comparacion)


Matriz manual: (100, 98167)
Matriz scikit-learn: (100, 98167)
Máxima diferencia absoluta: 0.0
Las matrices TF-IDF coinciden correctamente.


,Término,TF-IDF manual,TF-IDF scikit-learn
0,0,12.160883,12.160883
1,00,0.000000,0.000000
2,000,1.000000,1.000000
3,001,0.000000,0.000000
4,0014,0.000000,0.000000
5,0023,0.000000,0.000000
6,003,0.000000,0.000000
7,0035,0.000000,0.000000
8,004,0.000000,0.000000
9,0042,0.000000,0.000000


### 5. Representar consultas y calcular similitud coseno

Defina **cuatro consultas** que representen necesidades de información diferentes.

Para cada consulta:

1. Escriban qué información espera encontrar el usuario.
2. Apliquen el mismo preprocesamiento de los documentos.
3. Construyan su vector TF-IDF utilizando **el vocabulario y los IDF del corpus**.
4. No recalculen IDF incorporando la consulta como otro documento.

Utilice `transform` del vectorizador ajustado.

**Similitud coseno**

La similitud coseno compara la orientación de los vectores de consulta y documento:

$
\operatorname{sim}(q,d)
=\frac{q\cdot d}{\|q\|\|d\|}
=\frac{\sum_i q_i d_i}
{\sqrt{\sum_i q_i^2}\sqrt{\sum_i d_i^2}}
$

Implemente este cálculo mediante el producto punto y las magnitudes de los vectores. Luego, comparen los resultados con `sklearn.metrics.pairwise.cosine_similarity`, que calcula ese producto punto normalizado. [sklearn](https://sklearn.org/stable/modules/metrics.html?utm_source=chatgpt.com)

Para una combinación consulta-documento, muestren:

**Términos compartidos | Productos parciales | Producto punto | Normas | Similitud**

In [10]:
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse.linalg import norm

In [11]:
# Definir cuatro consultas y sus necesidades de información
consultas = {
    "Consulta 1": {
        "texto": "history of england",
        "necesidad": "Encontrar documentos relacionados con la historia de Inglaterra."
    },
    "Consulta 2": {
        "texto": "science and mathematics",
        "necesidad": "Encontrar documentos sobre ciencia y matemáticas."
    },
    "Consulta 3": {
        "texto": "love and friendship",
        "necesidad": "Encontrar documentos relacionados con el amor y la amistad."
    },
    "Consulta 4": {
        "texto": "war and military",
        "necesidad": "Encontrar documentos sobre guerras y temas militares."
    }
}

# Aplicar el mismo preprocesamiento del corpus
consultas_tokens = {
    nombre: tokenizar(datos["texto"])
    for nombre, datos in consultas.items()
}

# Construir los vectores TF-IDF con el vocabulario e IDF del corpus
nombres_consultas = list(consultas.keys())
tokens_consultas = list(consultas_tokens.values())

matriz_consultas = vectorizador.transform(tokens_consultas)

# Calcular similitud coseno manualmente
producto_punto = matriz_consultas @ matriz_tfidf.T

normas_consultas = norm(matriz_consultas, axis=1)
normas_documentos = norm(matriz_tfidf, axis=1)

denominador = np.outer(normas_consultas, normas_documentos)

similitud_manual = np.divide(
    producto_punto.toarray(),
    denominador,
    out=np.zeros(denominador.shape),
    where=denominador != 0
)

# Calcular similitud coseno con scikit-learn
similitud_sklearn = cosine_similarity(
    matriz_consultas,
    matriz_sklearn
)

# Comparar resultados
diferencia_coseno = np.max(
    np.abs(similitud_manual - similitud_sklearn)
)

print("Máxima diferencia entre similitudes:", diferencia_coseno)
print("¿Coinciden?:", np.allclose(
    similitud_manual, similitud_sklearn, atol=1e-10
))

# Mostrar consultas y necesidades de información
tabla_consultas = pd.DataFrame([
    {
        "Consulta": nombre,
        "Texto": datos["texto"],
        "Necesidad de información": datos["necesidad"],
        "Tokens": consultas_tokens[nombre]
    }
    for nombre, datos in consultas.items()
])

display(tabla_consultas)

# Analizar una combinación consulta-documento
indice_consulta = 0
indice_documento = 0

vector_q = matriz_consultas.getrow(indice_consulta)
vector_d = matriz_tfidf.getrow(indice_documento)

# Identificar términos compartidos
indices_compartidos = np.intersect1d(
    vector_q.indices,
    vector_d.indices
)

productos_parciales = []

for indice in indices_compartidos:
    peso_q = vector_q[0, indice]
    peso_d = vector_d[0, indice]

    productos_parciales.append({
        "Término": vocabulario_ordenado[indice],
        "Peso consulta": peso_q,
        "Peso documento": peso_d,
        "Producto parcial": peso_q * peso_d
    })

tabla_productos = pd.DataFrame(productos_parciales)

print("\nConsulta:", consultas[nombres_consultas[indice_consulta]]["texto"])
print("Documento:", nombres_documentos[indice_documento])

display(tabla_productos)

# Mostrar producto punto, normas y similitud
print("Producto punto:", producto_punto[indice_consulta, indice_documento])
print("Norma consulta:", normas_consultas[indice_consulta])
print("Norma documento:", normas_documentos[indice_documento])
print("Similitud manual:", similitud_manual[indice_consulta, indice_documento])
print("Similitud scikit-learn:", similitud_sklearn[indice_consulta, indice_documento])


Máxima diferencia entre similitudes: 1.4988010832439613e-15
¿Coinciden?: True


,Consulta,Texto,Necesidad de información,Tokens
0,Consulta 1,history of england,Encontrar documentos relacionados con la histo...,"[history, of, england]"
1,Consulta 2,science and mathematics,Encontrar documentos sobre ciencia y matemáticas.,"[science, and, mathematics]"
2,Consulta 3,love and friendship,Encontrar documentos relacionados con el amor ...,"[love, and, friendship]"
3,Consulta 4,war and military,Encontrar documentos sobre guerras y temas mil...,"[war, and, military]"



Consulta: history of england
Documento: cache/epub/10000/pg10000.txt


,Término,Peso consulta,Peso documento,Producto parcial
0,england,1.400478,50.417192,70.608147
1,of,1.000000,902.000000,902.000000


Producto punto: 972.608146923505
Norma consulta: 2.1739964283556326
Norma documento: 2061.5592026427767
Similitud manual: 0.21701175825457514
Similitud scikit-learn: 0.21701175825457528


### 6. Construir el ranking usando TF-IDF

Para cada consulta, ordenen los documentos de mayor a menor similitud coseno y presenten el **top-5**:

**Posición | Identificador | Título | Puntuación | Fragmento**

Las versiones explícita y basada en bibliotecas deben producir las mismas puntuaciones dentro de la tolerancia establecida.

In [12]:

# Función para obtener el ranking top-5
def obtener_ranking(indice_consulta, similitudes, top_k=5):
    puntuaciones = similitudes[indice_consulta]

    # Ordenar documentos de mayor a menor puntuación
    indices_ordenados = np.argsort(-puntuaciones, kind="stable")[:top_k]

    resultados = []

    for posicion, indice in enumerate(indices_ordenados, start=1):
        nombre = nombres_documentos[indice]

        # Recuperar el texto original
        texto_original = documentos_originales[nombre]

        # Limpiar saltos de línea para mostrar el fragmento
        fragmento = " ".join(texto_original.split())[:200]

        resultados.append({
            "Posición": posicion,
            "Identificador": nombre,
            "Título": Path(nombre).stem,
            "Puntuación": puntuaciones[indice],
            "Fragmento": fragmento
        })

    return pd.DataFrame(resultados)


# Obtener ranking para las cuatro consultas
rankings = {}

for i, nombre_consulta in enumerate(nombres_consultas):
    print("\n" + "=" * 70)
    print(nombre_consulta)
    print("Texto:", consultas[nombre_consulta]["texto"])
    print("Necesidad:", consultas[nombre_consulta]["necesidad"])

    # Ranking usando similitud manual
    ranking_manual = obtener_ranking(i, similitud_manual)

    # Ranking usando scikit-learn
    ranking_sklearn = obtener_ranking(i, similitud_sklearn)

    rankings[nombre_consulta] = ranking_manual

    print("\nTop-5 documentos recuperados:")
    display(ranking_manual)

    # Comparar puntuaciones
    diferencia = np.max(np.abs(
        ranking_manual["Puntuación"].values -
        ranking_sklearn["Puntuación"].values
    ))

    mismos_documentos = (
        ranking_manual["Identificador"].tolist() ==
        ranking_sklearn["Identificador"].tolist()
    )

    print("Máxima diferencia:", diferencia)
    print("¿Mismo orden de documentos?:", mismos_documentos)
    print("¿Puntuaciones equivalentes?:", diferencia < 1e-10)



Consulta 1
Texto: history of england
Necesidad: Encontrar documentos relacionados con la historia de Inglaterra.

Top-5 documentos recuperados:


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,cache/epub/10060/pg10060.txt,pg10060,0.223928,The Project Gutenberg eBook of Discourses: Bio...
1,2,cache/epub/10010/pg10010.txt,pg10010,0.217584,The Project Gutenberg eBook of The Eulogies of...
2,3,cache/epub/10065/pg10065.txt,pg10065,0.217047,The Project Gutenberg eBook of The Constitutio...
3,4,cache/epub/10000/pg10000.txt,pg10000,0.217012,The Project Gutenberg eBook of The Magna Carta...
4,5,cache/epub/10090/pg10090.txt,pg10090,0.214751,The Project Gutenberg eBook of Peaceless Europ...


Máxima diferencia: 5.828670879282072e-16
¿Mismo orden de documentos?: True
¿Puntuaciones equivalentes?: True

Consulta 2
Texto: science and mathematics
Necesidad: Encontrar documentos sobre ciencia y matemáticas.

Top-5 documentos recuperados:


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,cache/epub/10064/pg10064.txt,pg10064,0.129327,The Project Gutenberg eBook of Beltane the Smi...
1,2,cache/epub/10050/pg10050.txt,pg10050,0.123373,The Project Gutenberg eBook of A Little Pilgri...
2,3,cache/epub/10051/pg10051.txt,pg10051,0.115072,The Project Gutenberg eBook of The Little Pilg...
3,4,cache/epub/10063/pg10063.txt,pg10063,0.105410,The Project Gutenberg eBook of Happiness and M...
4,5,cache/epub/1001/pg1001.txt,pg1001,0.103297,"The Project Gutenberg eBook of Divine Comedy, ..."


Máxima diferencia: 3.3306690738754696e-16
¿Mismo orden de documentos?: True
¿Puntuaciones equivalentes?: True

Consulta 3
Texto: love and friendship
Necesidad: Encontrar documentos relacionados con el amor y la amistad.

Top-5 documentos recuperados:


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,cache/epub/10064/pg10064.txt,pg10064,0.227379,The Project Gutenberg eBook of Beltane the Smi...
1,2,cache/epub/10050/pg10050.txt,pg10050,0.213371,The Project Gutenberg eBook of A Little Pilgri...
2,3,cache/epub/10063/pg10063.txt,pg10063,0.200715,The Project Gutenberg eBook of Happiness and M...
3,4,cache/epub/10051/pg10051.txt,pg10051,0.199363,The Project Gutenberg eBook of The Little Pilg...
4,5,cache/epub/10006/pg10006.txt,pg10006,0.181413,The Project Gutenberg eBook of La Fiammetta Th...


Máxima diferencia: 5.551115123125783e-16
¿Mismo orden de documentos?: True
¿Puntuaciones equivalentes?: True

Consulta 4
Texto: war and military
Necesidad: Encontrar documentos sobre guerras y temas militares.

Top-5 documentos recuperados:


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,cache/epub/10064/pg10064.txt,pg10064,0.210298,The Project Gutenberg eBook of Beltane the Smi...
1,2,cache/epub/10050/pg10050.txt,pg10050,0.198787,The Project Gutenberg eBook of A Little Pilgri...
2,3,cache/epub/10051/pg10051.txt,pg10051,0.185211,The Project Gutenberg eBook of The Little Pilg...
3,4,cache/epub/10078/pg10078.txt,pg10078,0.174406,The Project Gutenberg eBook of Plain Words fro...
4,5,cache/epub/10063/pg10063.txt,pg10063,0.169793,The Project Gutenberg eBook of Happiness and M...


Máxima diferencia: 4.996003610813204e-16
¿Mismo orden de documentos?: True
¿Puntuaciones equivalentes?: True


## Acuerdo de uso de inteligencia artificial

Declaro que utilicé inteligencia artificial como apoyo para revisar la estructura del notebook, detectar inconsistencias entre las instrucciones y el código, proponer correcciones de reproducibilidad y mejorar la redacción del análisis. La IA no sustituye la comprensión ni la responsabilidad del estudiante: revisé las sugerencias y debo ejecutar el notebook, comprobar sus resultados, explicar el funcionamiento de los algoritmos y citar cualquier fuente externa utilizada. Asumo la autoría y responsabilidad académica del trabajo final y me comprometo a no presentar como verificados resultados que no haya reproducido.

**Justificación:** el uso fue de asistencia técnica y editorial para localizar errores y hacer explícitos los límites de las mediciones. La interpretación se apoya en los resultados guardados y en la complejidad esperada de los algoritmos; los datos de RAM y almacenamiento temporal no se inventaron. La ejecución final debe confirmarse en el entorno donde se entregará el notebook.